# Tournoi trinquet

Ce carnet ne contient plus de logique : tout vit dans le dépôt `tournoi-trinquet`, qui est testé.
Il ne reste ici que ce que Colab apporte vraiment — l'installation et l'authentification Google.

1. installer SWI-Prolog et la bibliothèque
2. s'authentifier auprès de Google Sheets
3. régler l'arbitrage aux curseurs
4. composer le planning, lire le bilan de chaque semaine et celui de la saison
5. le relire en tableau, puis l'emporter en CSV ou dans un classeur Google
6. consulter l'annuaire des joueurs

Le bilan de la cellule 4 est la première chose à lire : il dit, semaine par
semaine, combien de places sont pourvues, qui ne joue pas et pourquoi, et ce
que chaque entorse a coûté. Tout se termine par le bilan de la saison, qui
compte les places restées vides et les joueurs restés dehors en donnant chaque
fois la raison, et les cas qu'elle couvre — de quoi aller vérifier sur place
plutôt que croire le total.

Les curseurs de la cellule 3 permettent d'essayer d'autres compositions sans
toucher au code : ils fixent le prix de chaque entorse, et le solveur cherche
le planning le moins cher. Aucun ne peut lever une règle dure ; poussé à son
maximum, en revanche, un curseur en devient une — le critère n'est alors plus
à vendre, quitte à laisser des places vides. C'est ainsi que la compagnie du
créneau se règle : elle était une règle stricte, elle est devenue un curseur,
qu'on pousse au bout pour la retrouver telle quelle. La règle des équipiers,
qui ne se séparent pas, n'a en revanche pas de curseur du tout : elle se lit
en tête de la cellule 3, et s'édite dans les réglages du dépôt.

Pour modifier une règle, un groupe ou une paire d'équipiers, éditez
`src/tournoi/reglages/tournoi.toml` dans le dépôt, puis réinstallez : rien de tout cela
n'est écrit dans le code. Pour un essai sans rien pousser, écrivez un TOML dans
`/content` et passez son chemin à `charger_config`.

In [1]:
# 1. Installation. Une minute environ, à refaire à chaque démarrage de la machine Colab.
#
# Le dépôt étant privé, pip a besoin d'un jeton GitHub. Rangez-le dans les
# secrets Colab (icône clé, colonne de gauche) sous le nom GITHUB_TOKEN, avec
# l'accès en lecture à ce dépôt. Il n'est jamais affiché ci-dessous : la
# commande passe par subprocess pour qu'il ne finisse pas dans la sortie
# du carnet, qui se sauvegarde avec lui.

import json
import subprocess
import sys

DEPOT = 'adussarps/tournoi-trinquet'

# Un module déjà importé ne se remplace pas en cours de route : on retient
# l'état d'avant l'installation pour savoir s'il faudra redémarrer la session.
DEJA_CHARGE = 'tournoi' in sys.modules

!apt-get -qq install swi-prolog-nox > /dev/null

from google.colab import userdata

try:
    jeton = userdata.get('GITHUB_TOKEN')
except Exception as erreur:
    raise SystemExit(
        "Ajoutez un secret GITHUB_TOKEN dans Colab (icône clé, à gauche) : "
        "un jeton GitHub ayant accès en lecture au dépôt privé."
    ) from erreur

source = f'git+https://{jeton}@github.com/{DEPOT}.git'

# Pip tient une URL git pour satisfaite dès que la version installée porte le
# même numéro, or le nôtre ne bouge pas d'un commit à l'autre : sans second
# passage forcé, une machine déjà servie garderait le code de la veille en
# annonçant que tout est en place. --no-deps y limite le travail au paquet.
for options in (['-q'], ['-q', '--force-reinstall', '--no-deps']):
    installation = subprocess.run(
        ['pip', 'install', *options, source],
        capture_output=True,
        text=True,
    )
    if installation.returncode != 0:
        raise SystemExit(
            "Installation impossible. Vérifiez que le jeton donne bien accès au dépôt."
        )

if DEJA_CHARGE:
    raise SystemExit(
        "La bibliothèque est à jour, mais Python garde en mémoire celle chargée "
        "avant. Redémarrez la session (Exécution > Redémarrer la session), puis "
        "relancez cette cellule."
    )

from tournoi.solveur.pont import verifier_installation

verifier_installation()  # lève une erreur explicite si SWI-Prolog manque

# Quel commit tourne, dit une fois pour toutes : un résultat surprenant laisse
# sinon toujours planer le doute d'une machine restée sur le code de la veille.
# Seule l'empreinte est affichée, jamais l'URL — elle porte le jeton.
def empreinte_installee():
    # Le paquet porte le nom du depot -- « tournoi-trinquet » --, et non celui du
    # module qu'on importe. Demander « tournoi » ne trouvait rien, et l'empreinte
    # s'annoncait inconnue sur une installation parfaitement en place.
    try:
        from importlib.metadata import Distribution

        _, paquet = DEPOT.split('/')
        trace = Distribution.from_name(paquet).read_text('direct_url.json') or '{}'
        return json.loads(trace).get('vcs_info', {}).get('commit_id', 'inconnue')[:7]
    except Exception:
        return 'inconnue'


print(f'SWI-Prolog et la bibliothèque sont en place — version {empreinte_installee()}.')

SWI-Prolog et la bibliothèque sont en place — version 9e5ea82.


In [2]:
# @title 2. Authentification et lecture de la feuille { display-mode: "form" }
#
# Collez le lien du classeur tel que la barre d'adresse l'affiche : le `gid`
# qu'il porte désigne l'onglet, et sera lu comme tel. Un nom de classeur ou une
# clé nue font aussi l'affaire.
#
# Le lien proposé ouvre l'onglet « Vos disponibilités », celui qui annonce la
# salle et le groupe servi en priorité sur chaque créneau. Un autre onglet se
# lit aussi bien, mais ce qu'il ne dit pas, les réglages le devineront.
#
# La feuille est lue, jamais écrite : rien de ce qui suit ne peut l'abîmer.

feuille = 'https://docs.google.com/spreadsheets/d/13cpakSx4QYCahwvasWnx7oTvT3XPOmQQJ5_U4Jk_Kh0/edit?gid=1334219092'  # @param {type:"string"}

from tournoi import sheets
from tournoi.config import charger_config, charger_config_equipes
from tournoi.extraction import extraire
from tournoi.rapports import effectifs as rapport_effectifs

config = charger_config()
grille = sheets.lire_grille(feuille)  # déclenche l'authentification Colab
extraction = extraire(grille, config)
donnees = extraction.donnees

for anomalie in extraction.anomalies:
    print(anomalie)

# Le vivier de chaque semaine, qui est le chiffre à lire avant de chercher : il
# dit ce que la feuille a vraiment répondu. Personne, et la semaine n'est pas
# encore ouverte ; tout le monde sur tous les créneaux, et elle n'a pas été
# remplie -- des cases cochées d'avance ne promettent rien, et le planning qui
# en sortira n'apprendra rien non plus. L'extraction le dit aussi, plus haut.
#
# Le détail par groupe est là parce que le total ne suffit pas : un créneau
# réservé au E ne se remplit qu'avec des E, et trois places vides faute de E
# étaient écrites dans la feuille, non manquées par la recherche.
for ligne in rapport_effectifs.lignes(donnees, config):
    print(ligne)

[groupe_prioritaire] '1S' n'est pas un groupe connu (C6a, C6b, C6c, C6d) : ces creneaux ne sont pas programmes
[compte] la feuille annonce 28 joueurs en 'B', nous en lisons 27 : une croix mal placee, ou une colonne mal lue
[semaine] semaine 3 laissee de cote : personne ne s'y est inscrit
[semaine] semaine 4 laissee de cote : personne ne s'y est inscrit
86 joueurs, 42 creneaux
  semaine 1 : 57 disponibles (11 A, 9 AB, 17 B, 13 C, 3 D, 4 E), 20 creneaux a composer et 1 hors tournoi
  semaine 2 : 58 disponibles (11 A, 10 AB, 16 B, 13 C, 4 D, 4 E), 20 creneaux a composer et 1 hors tournoi


In [3]:
# @title 3. Réglages { display-mode: "form" }

# @markdown ### Comment lire ces curseurs
# @markdown Chacun est un **effort** demandé au solveur, et tous vont dans le
# @markdown même sens : **plus vous montez, plus il en tient compte**. Monter
# @markdown `eviter_un_joueur_qui_ne_joue_pas`, c'est donc obtenir **moins** de
# @markdown joueurs laissés sur la touche. Descendre à zéro, c'est dire que le
# @markdown critère est indifférent.

# @markdown Ce sont des efforts *relatifs* : seul leur rapport compte. Éviter
# @markdown une équipe non mélangée à 20 contre une partie irrégulière à 15 dit
# @markdown qu'entre les deux, on préfère défaire la seconde. Tout doubler ne
# @markdown change rien ; n'en doubler qu'un change tout.

# @markdown Aucun curseur ne peut lever une règle dure — les postes, **une seule
# @markdown partie par jour et par joueur**, les croisements interdits, les
# @markdown féminines par paires, les trois parties maximum dans la semaine et
# @markdown les quotas demandés valent toujours. Remplir non plus n'est pas
# @markdown négociable : le solveur pourvoit d'abord le plus de places possible,
# @markdown puis compose au mieux à ce niveau-là. Aucun curseur n'achètera donc
# @markdown un trou contre une belle partie.

# @markdown ### La règle du club, qui ne se règle pas
# @markdown **La paire d'équipiers.** Deux équipiers disponibles sur le même
# @markdown créneau y jouent tous les deux ou ni l'un ni l'autre, jusqu'à leur
# @markdown deuxième partie de la semaine ; la troisième, celle qui comble, se
# @markdown joue avec qui veut. Et quand ils jouent tous les deux, ils jouent
# @markdown **dans la même équipe** — on ne fait pas venir une paire pour
# @markdown l'installer de part et d'autre du filet. Le curseur qui en faisait un
# @markdown encouragement a disparu : c'est une règle, pas un arbitrage. Elle
# @markdown s'édite dans `tournoi.toml`, comme les groupes et les compagnies.

# @markdown **Faire jouer tout le monde** se décide de la même façon, juste
# @markdown après : à remplissage égal, le solveur fait asseoir le plus de
# @markdown joueurs différents possible, et aucun mélange évité ne peut plus
# @markdown renvoyer quelqu'un chez lui. Ceux qui restent dehors malgré tout
# @markdown sont nommés dans le bilan de la cellule 4, avec ce qui leur a barré
# @markdown chacun de leurs créneaux.

# @markdown ### Un curseur poussé au bout devient une règle
# @markdown Un prix, même très élevé, reste un prix : il se trouvera toujours une
# @markdown place vide, plus chère encore, pour l'acheter. Aussi le **maximum
# @markdown (200)** veut dire autre chose : **plus à vendre**. Le solveur
# @markdown laissera plutôt le créneau incomplet, et le bilan annonce alors
# @markdown « aucun prix ne l'achète ». À manier avec prudence : chaque curseur
# @markdown poussé au bout se paie en places vides.

# @markdown Les deux premiers de la liste font exception : remplir les créneaux
# @markdown et faire jouer tout le monde se décident déjà avant les prix, ce qui
# @markdown est plus fort qu'un interdit ; et en faire des règles rendrait la
# @markdown plupart des semaines insolubles, faute d'un planning parfait. Les
# @markdown pousser au bout ne change donc rien.

# @markdown ---
# @markdown #### Ce qu'on cherche à éviter
# @markdown Le premier ne décide plus de rien : les joueurs sans partie se
# @markdown règlent avant les autres critères. Il ne sert qu'à chiffrer ce qu'ils
# @markdown coûtent dans le bilan. Les deux croisements, eux, ne valent que là où
# @markdown aucune compagnie n'est déclarée : les créneaux du `C` et ceux qui
# @markdown n'annoncent aucune réservation. Ailleurs, la compagnie a déjà dit qui
# @markdown entre.
eviter_un_joueur_qui_ne_joue_pas = 60  # @param {type:"slider", min:0, max:200, step:5}
eviter_de_melanger_des_groupes_qui_ne_vont_pas_ensemble = 40  # @param {type:"slider", min:0, max:200, step:5}
eviter_de_melanger_le_D_et_le_E = 40  # @param {type:"slider", min:0, max:200, step:5}
eviter_deux_parties_a_un_joueur_pres = 25  # @param {type:"slider", min:0, max:200, step:5}
eviter_une_equipe_non_melangee = 20  # @param {type:"slider", min:0, max:200, step:5}
eviter_une_partie_ni_4_0_ni_2_2 = 15  # @param {type:"slider", min:0, max:200, step:5}
eviter_une_troisieme_partie_dans_la_semaine = 10  # @param {type:"slider", min:0, max:200, step:5}

# @markdown #### La compagnie du créneau
# @markdown Un créneau réservé se remplit par cercles : d'abord le groupe qu'il
# @markdown sert, puis sa compagnie — `E` et `D` ensemble, `B` avec `D`, `AB`
# @markdown avec `A` —, puis, si les places ne sont pas pourvues, n'importe qui.
# @markdown Ce dernier cercle est ce que ce curseur tarife. Les créneaux du `C`
# @markdown et ceux marqués « TOUS » ne sont jamais concernés, et les compagnies
# @markdown s'éditent dans `tournoi.toml`, où vider la table les supprime.

# @markdown Le club l'avait d'abord voulue stricte, et la pousser au maximum la
# @markdown rend telle quelle : plus personne d'autre, quitte à laisser la place
# @markdown vide. C'est mesuré, à temps égal sur la feuille de la saison : 199
# @markdown places pourvues sur 240 au maximum, contre 239 au prix proposé.
# @markdown Chaque fois qu'on élargit, le bilan dit qui est entré et faute de
# @markdown qui. Le CSV — et le Google Sheet — le répètent dans la colonne
# @markdown « elargissement », dès qu'on quitte le groupe réservé, pas seulement
# @markdown au-delà de sa compagnie.
eviter_d_asseoir_quelqu_un_hors_de_la_compagnie = 40  # @param {type:"slider", min:0, max:200, step:5}

# @markdown #### La ligne « catégorie prioritaire » de la feuille
# @markdown Des cases, et non un curseur : mesure faite, un prix intermédiaire n'y
# @markdown changeait rien. Les places pourvues passant avant tout, le solveur
# @markdown trouvait toujours un planning aussi rempli qui servait aussi mal les
# @markdown réservations — 22 places prises sur la première semaine, quel que soit
# @markdown le prix et le temps laissé. Ce n'était pas le prix qu'il fallait
# @markdown changer, mais l'ordre de la recherche.
# @markdown
# @markdown **La première case est le mode proposé** : le solveur remplit d'abord
# @markdown chaque créneau avec le seul groupe qu'il sert, puis revient élargir
# @markdown ceux qui sont restés incomplets — la compagnie d'abord, les autres
# @markdown ensuite. C'est la feuille lue créneau par créneau, comme au club, et
# @markdown cela ne coûte pas de place : ce que le groupe servi ne remplissait
# @markdown pas, la seconde passe le prend. La recherche ne dure pas plus
# @markdown longtemps : la passe stricte prend la majorité du temps de
# @markdown remplissage — c'est elle qui compte — et ce qu'elle n'a pas
# @markdown consommé revient à l'élargissement.
# @markdown
# @markdown **La seconde va plus loin et ferme la porte** : les créneaux réservés
# @markdown n'accueillent alors plus personne d'autre, quitte à rester incomplets
# @markdown — sur la première semaine, huit places vides de plus. Les créneaux
# @markdown marqués « TOUS » ne sont jamais concernés, et le bilan compte les
# @markdown places prises dans tous les cas.
servir_d_abord_le_groupe_reserve = True  # @param {type:"boolean"}
n_accueillir_personne_d_autre = False  # @param {type:"boolean"}

# @markdown #### Temps de recherche
# @markdown Par semaine, en secondes de vraie horloge, et il est tenu : chaque
# @markdown étape s'arrête à son échéance. C'est un plafond, pas une attente
# @markdown promise : la recherche rend la main dès qu'elle n'a plus rien à
# @markdown gagner. Le temps sert d'abord à remplir.
# @markdown
# @markdown Deux minutes suffisent à une feuille normalement remplie : sur celle
# @markdown de la saison, une cinquantaine de disponibles par semaine, elles
# @markdown pourvoient 237 places sur 240, et dix secondes en pourvoient déjà
# @markdown 234. Deux cas demandent beaucoup plus, et ils se cumulent : une
# @markdown feuille dont toutes les cases sont cochées — chacun étant éligible
# @markdown partout, il y a le maximum de monde à départager sur chaque place —
# @markdown et une machine lente, Colab en donnant une trois fois plus lente
# @markdown qu'un portable récent. Le temps par tentative étant une part du
# @markdown budget, c'est en montant le curseur qu'on rend à la machine lente
# @markdown les secondes qui lui manquent. Le bilan dit toujours combien de
# @markdown places sont restées vides.
secondes_de_reflexion_par_semaine = 300  # @param {type:"slider", min:10, max:3600, step:30}

from dataclasses import replace

from tournoi.config import PRIX_MAXIMUM

# Un croisement se déclare par paire de groupes, si bien qu'une même règle en
# produit plusieurs ; chacun porte donc le nom du réglage dont il dépend, et
# c'est par ce nom qu'un curseur les prend tous à la fois. Un croisement dont le
# nom manquerait ici garderait le prix inscrit dans les règles.
efforts = {
    'des_groupes_qui_ne_vont_pas_ensemble': eviter_de_melanger_des_groupes_qui_ne_vont_pas_ensemble,
    'deux_niveaux_du_meme_groupe': eviter_de_melanger_le_D_et_le_E,
}

# Le solveur, lui, raisonne en coûts : ce qu'on cherche à obtenir y est un coût
# négatif. La conversion se fait ici, une fois, pour que les curseurs restent
# tous dans le même sens -- sans quoi il y en aurait un à monter pour en avoir
# moins et un autre à descendre pour en avoir plus.
config = replace(
    config,
    poids={
        **config.poids,
        'joueur_sans_partie': eviter_un_joueur_qui_ne_joue_pas,
        'trio_commun': eviter_deux_parties_a_un_joueur_pres,
        'equipe_homogene': eviter_une_equipe_non_melangee,
        'composition_irreguliere': eviter_une_partie_ni_4_0_ni_2_2,
        'place_hors_compagnie': eviter_d_asseoir_quelqu_un_hors_de_la_compagnie,
        'partie_supplementaire': eviter_une_troisieme_partie_dans_la_semaine,
        # La réservation garde le prix des règles : c'est lui qui, à remplissage
        # égal, préfère encore le groupe servi. Le mettre à zéro -- ce que faisait
        # ce carnet -- rendait la ligne « catégorie prioritaire » gratuite une
        # fois la passe stricte terminée. Refuser tout élargissement est autre
        # chose : le critère n'est plus à vendre.
        **({'place_hors_priorite': PRIX_MAXIMUM} if n_accueillir_personne_d_autre else {}),
    },
    melanges_deconseilles=tuple(
        replace(melange, penalite=efforts.get(melange.reglage, melange.penalite))
        for melange in config.melanges_deconseilles
    ),
    servir_d_abord_la_reservation=servir_d_abord_le_groupe_reserve,
)

# Les valeurs sont absolues, pas des écarts : relancer la cellule deux fois de
# suite donne le même réglage, et repartir de zéro se fait en relançant la 2.
print(f'Réglage retenu, {secondes_de_reflexion_par_semaine} s de recherche par semaine :')
for critere, prix in sorted(config.poids.items(), key=lambda couple: -couple[1]):
    if config.interdit(critere):
        # Le curseur est au bout : le critère a quitté l'arbitrage et le solveur
        # ne l'achètera plus, quitte à laisser une place vide.
        sens = 'REFUSÉ, aucun prix ne l’achète'
    elif critere in ('place_vide', 'joueur_sans_partie'):
        # Ceux-là se règlent par paliers, avant que le coût n'entre en jeu : leur
        # prix chiffre le planning, il ne départage jamais deux plannings.
        sens = 'hors négociation'
    else:
        sens = 'à éviter' if prix > 0 else 'à rechercher' if prix < 0 else 'indifférent'
    print(f'  {critere:>28} : {abs(prix):>3}  {sens}')

# Les croisements déconseillés ne sont pas des poids comme les autres : ils sont
# portés par la règle elle-même, et manqueraient au récapitulatif. Les groupes y
# sont nommés comme partout ailleurs, un groupe masqué prenant le nom de celui
# qui le couvre -- deux croisements peuvent alors s'écrire pareil.
croisements = {}
for melange in config.melanges_deconseilles:
    noms = sorted({config.groupe_affiche(groupe) for groupe in melange.groupes})
    libelle = f'deux niveaux du {noms[0]}' if len(noms) == 1 else '+'.join(noms)
    croisements[libelle] = max(croisements.get(libelle, 0), melange.penalite)

# Un croisement au maximum ne se paie plus : il rejoint les mélanges interdits,
# et ne se produira pas, même s'il faut laisser des places vides.
for libelle, prix in croisements.items():
    sens = 'REFUSÉ, aucun prix ne l’achète' if prix >= PRIX_MAXIMUM else 'à éviter'
    print(f'  {libelle + " ensemble":>28} : {prix:>3}  {sens}')

Réglage retenu, 300 s de recherche par semaine :
                    place_vide : 100  hors négociation
            joueur_sans_partie :  60  hors négociation
          place_hors_compagnie :  40  à éviter
                   trio_commun :  25  à éviter
               equipe_homogene :  20  à éviter
       composition_irreguliere :  15  à éviter
           place_hors_priorite :  15  à éviter
         partie_supplementaire :  10  à éviter
                 A+AB ensemble :  40  à éviter


In [4]:
# 4. Planification, avec les réglages de la cellule précédente.
#
# Chaque semaine s'ouvre sur son bilan : les places pourvues, les joueurs qui
# jouent et ceux qui restent dehors — ces derniers nommés, avec la raison pour
# chacun de leurs créneaux — puis le compte de chaque entorse et son prix.
#
# Sous chaque créneau s'affiche ensuite ce qui a décidé de sa composition, la
# raison de chaque place vide, puis les remplaçants à appeler si quelqu'un se
# décommande. Passez explications=False ou suppleants=False pour alléger.
#
# La sortie se termine par le bilan de la saison : les places restées vides et
# les joueurs restés dehors, rangés par raison, avec les créneaux et les noms
# concernés. Deux raisons s'y annoncent « rattrapables à la main » — une place
# qu'un joueur pouvait prendre, un absent à qui quelqu'un pouvait céder la
# sienne : ce sont les seules sur lesquelles vous pouvez encore agir.
#
# Le temps sert d'abord à remplir, et la suite ne fait plus qu'embellir. Deux
# minutes par semaine suffisent à une feuille normalement remplie ; le réglage
# proposé en laisse cinq, et le curseur monte à dix pour chercher pendant le
# café. Une barre annonce l'attente, et chaque semaine dit ce qu'elle a rendu
# dès qu'elle est finie — sans attendre le bilan complet.
import threading
import time

from tqdm.auto import tqdm

from tournoi.rapports import planning as rapport_planning
from tournoi.solveur.pont import resoudre

budget = secondes_de_reflexion_par_semaine
barre = tqdm(total=budget * len(tuple(donnees.semaines())), unit='s', desc='Recherche')
acquis = 0  # secondes créditées par les semaines déjà terminées
termine = threading.Event()


def suivre_l_horloge():
    # La barre avance sur l'horloge : le budget étant tenu, une semaine ne
    # dépasse pas sa part, et celle qui aboutit plus tôt fait sauter la barre.
    depart = time.monotonic()
    while not termine.wait(1):
        barre.n = min(int(max(acquis, time.monotonic() - depart)), barre.total)
        barre.refresh()


def annoncer(etape):
    global acquis
    if etape.resultat is None:
        barre.set_description(f'Semaine {etape.semaine} ({etape.rang}/{etape.total})')
        return
    faits = etape.resultat.planning.statistiques()
    barre.write(
        f'semaine {etape.semaine} : '
        f'{faits.places_pourvues}/{faits.places} places pourvues'
    )
    acquis = etape.rang * budget
    barre.n = min(acquis, barre.total)
    barre.refresh()


horloge = threading.Thread(target=suivre_l_horloge, daemon=True)
horloge.start()
try:
    planning = resoudre(donnees, config, limite_par_semaine=budget, annoncer=annoncer)
finally:
    termine.set()
    barre.n = barre.total
    barre.close()

for ligne in rapport_planning.en_console(planning, donnees, config):
    print(ligne)

Recherche:   0%|          | 0/600 [00:00<?, ?s/s]

semaine 1 : 76/80 places pourvues
semaine 2 : 77/80 places pourvues

Semaine 1 — 76/80 places pourvues sur 20 creneaux, 55 joueurs sur 57 disponibles
  joueurs sans partie : 2 (C.TONICELLO, J.LAURENT) (+120)
  parties par joueur : 38 en jouent 1, 13 en jouent 2, 4 en jouent 3
  places vides : 4 (+400)
  A+AB : 1 (+40)
  equipes non melangees : 2 (+40)
  compositions ni 4+0 ni 2+2 : 5 (+75)
  places prises sur le groupe qui avait la priorite : 26 (+390)
  places elargies au-dela de la compagnie du creneau : 9 (+360)
  parties au-dela du quota habituel : 4 (+40)
  parties qui se ressemblent : 0 (+0)
  equipiers reunis : 3 (regle du club, sans prix)
  prix total de la semaine : 1465
  C.TONICELLO (B av&ar) ne joue pas
      C5a : complet, et il jouerait sans T.LOUSTALET, son equipier disponible
      C10a : complet, et il laisserait un joueur seul pour le groupe C
      C11a : complet, et il jouerait sans T.LOUSTALET, son equipier disponible
  J.LAURENT (AB av) ne joue pas
      C11a : co

In [17]:
# 5. Le planning en tableau, à emporter en CSV ou en classeur Google.
#
# Soixante-trois lignes : Colab l'affiche sans peine. La colonne « explication »
# recopie ce que le rapport dit sous le créneau ; « remplacants » donne les
# premiers noms à appeler, dans l'ordre où il vaut mieux les appeler ;
# « disponibles » donne le vivier entier du créneau, règles mises de côté.
import ipywidgets as widgets
import pandas as pd
import re

from google.colab import files

# Define the superscript map and conversion function
SUPERSCRIPT_MAP = {
    'A': 'ᴬ', 'B': 'ᴮ', 'C': 'ᶜ', 'D': 'ᴰ', 'E': 'ᴱ', 'F': 'ᶠ', 'G': 'ᴳ', 'H': 'ᴴ',
    'I': 'ᴵ', 'J': 'ᴶ', 'K': 'ᴷ', 'L': 'ᴸ', 'M': 'ᴹ', 'N': 'ᴺ', 'O': 'ᴼ', 'P': 'ᴾ',
    'Q': 'ᵠ', 'R': 'ᴿ', 'S': 'ˢ', 'T': 'ᵀ', 'U': 'ᵁ', 'V': 'ⱽ', 'W': 'ᵂ', 'X': 'ˣ',
    'Y': 'ʸ', 'Z': 'ᶻ',
    '0': '⁰', '1': '¹', '2': '²', '3': '³', '4': '⁴', '5': '⁵', '6': '⁶', '7': '⁷',
    '8': '⁸', '9': '⁹',
}

def to_superscript(text):
    return ''.join(SUPERSCRIPT_MAP.get(char, char) for char in text.upper())

# This function will process a single cell content, which might contain multiple player entries
def process_player_string_in_cell(cell_content):
    if not isinstance(cell_content, str) or not cell_content:
        return cell_content

    # Split the cell content by ' / ' to process each player separately
    parts = [part.strip() for part in cell_content.split(' / ')]
    transformed_parts = []

    for part in parts:
        # First, try to match the overall structure: 'Name (content_in_parenthesis)'
        outer_match = re.match(r"^(.*?)\s*\((.+?)\)$", part)
        if outer_match:
            name_part = outer_match.group(1).strip()
            content_in_parenthesis = outer_match.group(2).strip()

            # Now, try to parse 'content_in_parenthesis' as 'Group Position'
            inner_match = re.match(r"^([\w]+)\s+(av|ar|av&ar)$", content_in_parenthesis)
            if inner_match:
                group = inner_match.group(1)
                position = inner_match.group(2)
                superscript_group = to_superscript(group)
                transformed_parts.append(f"{name_part}{superscript_group} ({position})")
            else:
                # If inner content does not match 'Group Position', check if it's just 'Position'
                if content_in_parenthesis in ("av", "ar", "av&ar"):
                    transformed_parts.append(f"{name_part} ({content_in_parenthesis})")
                else:
                    # If inner content does not match expected patterns, keep original outer match format
                    transformed_parts.append(f"{name_part} ({content_in_parenthesis})")
        else:
            # If no outer parenthesis match, keep the original part
            transformed_parts.append(part)

    return ' / '.join(transformed_parts)

tableau = pd.DataFrame(
    rapport_planning.lignes(planning, donnees, config),
    columns=rapport_planning.ENTETE_CSV,
)

# Columns that likely contain player names in the specified format
player_columns = [
    'avant1', 'arriere1', 'avant2', 'arriere2', 'avant3', 'arriere3',
    'remplacants', 'disponibles'
]

for col in player_columns:
    if col in tableau.columns:
        tableau[col] = tableau[col].apply(process_player_string_in_cell)

display(tableau)

CHEMIN_CSV = rapport_planning.ecrire_csv('/content/planning.csv', planning, donnees, config)

# Le fichier vit sur la machine Colab, qui disparaît en fin de séance : le
# bouton en fait une copie sur votre ordinateur. On peut le presser autant de
# fois qu'on veut, et il reste utilisable tant que la cellule est affichée.
# Il s'ouvre tel quel dans Excel, accents compris.
telechargement = widgets.Button(
    description='Télécharger le CSV',
    icon='download',
    button_style='primary',
)

# L'autre sortie possible : un classeur Google. Il est créé pour l'occasion dans
# votre Drive, et le classeur des disponibilités n'y est pour rien -- il reste
# lu et jamais écrit. Une fois créé, le bouton devient le lien qui l'ouvre.
tableur = widgets.Button(
    description='Ouvrir dans Google Sheets',
    icon='table',
)
sortie = widgets.Output()


def telecharger(_):
    with sortie:
        files.download(str(CHEMIN_CSV))


STYLE_LIEN = (
    'display:inline-block;padding:5px 14px;margin-left:4px;border:1px solid #bbb;'
    'border-radius:4px;text-decoration:none;font-size:13px;line-height:20px'
)


def vers_google_sheets(bouton):
    from datetime import datetime

    from tournoi import sheets

    # Tout se passe sous `sortie`, y compris ce qui échoue : une erreur levée
    # hors de cette zone n'apparaîtrait nulle part, le bouton restant muet.
    with sortie:
        bouton.disabled = True
        bouton.description = 'Création en cours…'
        try:
            lien = sheets.creer_classeur(
                rapport_planning.en_grille(planning, donnees, config),
                f'Planning trinquet du {datetime.now():%d/%m/%Y à %Hh%M}',
            )
        finally:
            bouton.disabled = False
            bouton.description = 'Ouvrir dans Google Sheets'

    # Le bouton laisse la place au lien : le classeur existe désormais, et le
    # presser à nouveau n'en créerait qu'un doublon. Relancer la cellule redonne
    # le bouton, et donc de quoi comparer deux réglages en deux classeurs.
    boutons.children = (
        telechargement,
        widgets.HTML(
            f'<a href="{lien}" target="_blank" style="{STYLE_LIEN}">Ouvrir le classeur créé</a>'
        ),
    )


telechargement.on_click(telecharger)
tableur.on_click(vers_google_sheets)
boutons = widgets.HBox([telechargement, tableur])
display(widgets.VBox([boutons, sortie]))

# Si le navigateur refusait le téléchargement, le fichier reste accessible par
# le panneau « Fichiers » (icône dossier, à gauche), à l'emplacement ci-dessous.
print(f'Écrit dans {CHEMIN_CSV}')

,semaine,jour,date,heure,lieu,creneau,priorite,avant1,arriere1,avant2,arriere2,statut,elargissement,explication,remplacants,disponibles
0,1,lundi,lundi 31 août,19h,CLERMONT,C1a,B,E.DE#BAILLENCOURTᴬ (av&ar),A.LACOSTEAnᴬᴮ (av&ar),T.CHOURREᴬᴮ (av&ar),A.LADAGNOUSᴬ (av&ar),complet,"E.DE#BAILLENCOURT (A), A.LACOSTEAn (AB), T.CHO...","priorite au groupe B non servie, aucun n'y jou...","J.MOREL (A av&ar), N.MATEOSNi (AB av&ar), R.AM...","A.MATEOSAm (C av), D.ZIZA (A av&ar), G.GAYET (..."
1,1,lundi,lundi 31 août,19h,JAB,C2a,A,D.BERBONᴬ (av),P.CASAUBIEILHᴬ (av&ar),G.TAUPIACᴬ (av&ar),G.GAYETᴬ (av&ar),complet,,priorite au groupe A : 4 des 4 places pourvues...,S.JOURDON#BLEUSEᴬ (ar),"A.LACOSTEAn (AB av&ar), A.LADAGNOUS (A av&ar),..."
2,1,lundi,lundi 31 août,20h,JAB,C3a,TOUS,R.CAZENAVEJuᴮ (av&ar),E.FORSANSᶜ (ar),S.GOSSELINᶜ (av),R.MADECᴮ (ar),complet,,creneau ouvert a tous les groupes | R.CAZENAVE...,"I.BIDEGAIN (E av&ar), L.ESTEBENET (C av&ar), P...","A.LADAGNOUS (A av&ar), D.BERBON (A av), D.ZIZA..."
3,1,mardi,mardi 1 septembre,18h,CdP,C4a,TOUS,P.INCHAURRAGAᴱ (av&ar),T.NEHRᴮ (ar),T.RODRIGUEZᴮ (av&ar),E.MOUSUTEGUYᴮ (ar),complet,,creneau ouvert a tous les groupes | compositio...,,"A.ESPILONDO (C av), D.ZIZA (A av&ar), E.DE#BAI..."
4,1,mardi,mardi 1 septembre,19h,CLERMONT,C5a,D,C.GUIRAUDᴰ (av),J.LARRICQᴰ (ar),S.DUSSARPSᴮ (av&ar),E.CANIᴮ (av&ar),complet,S.DUSSARPS (B) et E.CANI (B) a la place du gro...,priorite au groupe D : 2 des 4 places pourvues...,"B.BOUCHETAL (B ar), P.AUGÉ (B ar)","A.LACOSTEAn (AB av&ar), A.LADAGNOUS (A av&ar),..."
5,1,mardi,mardi 1 septembre,19h,CdP,C6a,1S,,,,,reserve au 1S,,,,"A.LACOSTEAn (AB av&ar), A.LADAGNOUS (A av&ar),..."
6,1,mardi,mardi 1 septembre,20h,CdP,C7a,AB,L.BOUEᴬ (av),T.CHOURREᴬᴮ (av&ar),R.AMROUCHEᴬᴮ (av&ar),S.BARALDIᴬᴮ (ar),complet,L.BOUE (A) a la place du groupe AB : aucun AB ...,priorite au groupe AB : 3 des 4 places pourvue...,"E.DE#BAILLENCOURT (A av&ar), D.BERBON (A av), ...","A.GAYMU (C ar), A.LADAGNOUS (A av&ar), D.BERBO..."
7,1,mardi,mardi 1 septembre,21h,CdP,C8a,AB,D.ZIZAᴬ (av&ar),A.LADAGNOUSᴬ (av&ar),G.GAYETᴬ (av&ar),M.BERENGUERᴬ (ar),complet,"D.ZIZA (A), A.LADAGNOUS (A), G.GAYET (A) et M....","priorite au groupe AB non servie, aucun n'y jo...",S.JOURDON#BLEUSEᴬ (ar),"A.GAYMU (C ar), G.TAUPIAC (A av&ar), R.AMROUCH..."
8,1,mercredi,mercredi 2 septembre,18h,CdP,C9a,TOUS,R.CAZENAVEJuᴮ (av&ar),T.NEHRᴮ (ar),T.RODRIGUEZᴮ (av&ar),P.AUGÉᴮ (ar),complet,,creneau ouvert a tous les groupes | R.CAZENAVE...,P.INCHAURRAGAᴱ (av&ar),"A.MATEOSAm (C av), D.BERBON (A av), D.ZIZA (A ..."
9,1,mercredi,mercredi 2 septembre,20h,CLERMONT,C10a,C,A.SAINT#GUIRONSᶜ (av),T.LOUSTALETᴮ (av&ar),E.GUEDOTᶜ (av),L.CAHUZACᴮ (ar),complet,T.LOUSTALET (B) et L.CAHUZAC (B) a la place du...,priorite au groupe C : 2 des 4 places pourvues...,"A.MATEOSAm (C av), E.PERRAIN (C av), L.CHOHOBI...","A.GAYMU (C ar), A.MATEOSAm (C av), C.TONICELLO..."


Écrit dans /content/planning.csv


In [15]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=tableau)

https://docs.google.com/spreadsheets/d/1mP8CKk82ZsEvnd4t3Q6mAdUFotoQo4QtFTbbWIZSWUw/edit#gid=0


In [6]:
# 6. L'annuaire : nom, prénom, postes tenables, groupe, et parties de la saison.
from tournoi.rapports import annuaire

display(
    pd.DataFrame(
        annuaire.lignes(donnees, config, planning),
        columns=annuaire.ENTETE_CSV,
    )
)

,nom,prenom,abrege,poste,postes,groupe,parties
0,BERBON,David,D.BERBON,av,avant,A,2
1,BERENGUER,Maïder,M.BERENGUER,ar,arriere,A,2
2,BOUE,Laetitia,L.BOUE,av,avant,A,5
3,CASAUBIEILH,Paul,P.CASAUBIEILH,av&ar,avant ou arriere,A,2
4,DE#BAILLENCOURT,Emilie,E.DE#BAILLENCOURT,av&ar,avant ou arriere,A,2
...,...,...,...,...,...,...,...
81,INCHAURRAGA,Patrick,P.INCHAURRAGA,av&ar,avant ou arriere,E,2
82,JAUREGUIBERRYph,Philippe,P.JAUREGUIBERRYph,av&ar,avant ou arriere,E,1
83,MARIAUD,Quentin,Q.MARIAUD,ar,arriere,E,0
84,OLIVEIRA,Frédérick,F.OLIVEIRA,av,avant,E,1


## Les autres rapports

Décommentez celui qui vous intéresse.

In [7]:
from tournoi.rapports import dispos, matchs_possibles, remplacants

# Disponibilités par créneau et par groupe :
# for ligne in dispos.rapport(donnees, config, semaine=1):
#     print(ligne)

# Équipes disponibles et matchs encore jouables :
# for ligne in matchs_possibles.rapport(donnees, charger_config_equipes()):
#     print(ligne)

# Remplaçants possibles, partie par partie, du plus indiqué au moins indiqué :
# for ligne in remplacants.rapport(planning, donnees, config, semaine=1):
#     print(ligne)

# Les mêmes rapports en levant le masque, c'est-à-dire en nommant les niveaux
# tels qu'ils sont plutôt que sous le nom qui les couvre. À garder pour vous, et
# à ne pas enregistrer avec la sortie : ce carnet est public.
# for ligne in dispos.rapport(donnees, config.sans_masque(), semaine=1):
#     print(ligne)